# Does age banding change fraud detection and age-group disparities?

Train three XGBoost models on the same simulated transactions using exact age, an age band, or neither age feature. Compare them on the same held-out transactions with the same number of alerts. Run this notebook with pandas, NumPy, scikit-learn, and XGBoost installed and the project's `data/fraudTrain.csv` in place.

The age-band scorer can accept a category directly. Age banding reduces the precision of the age data provided for scoring. Other inputs can still correlate with age. This experiment measures age-group differences on this dataset. A formal differential privacy guarantee would require a separate mechanism.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

path = Path("data/fraudTrain.csv")
if not path.is_file():
    path = Path("../data/fraudTrain.csv")

transactions = pd.read_csv(
    path,
    usecols=[
        "trans_date_trans_time", "dob", "amt", "category", "lat", "long",
        "merch_lat", "merch_long", "is_fraud",
    ],
    parse_dates=["trans_date_trans_time", "dob"],
)
timestamp = transactions["trans_date_trans_time"]
transactions["hour"] = timestamp.dt.hour
transactions["day_of_week"] = timestamp.dt.day_name()
# Match the age calculation used by the existing XGBoost baseline.
transactions["age"] = ((timestamp - transactions["dob"]).dt.days / 365.25).astype(int)

lat, lon, merchant_lat, merchant_lon = (
    np.radians(transactions[column])
    for column in ("lat", "long", "merch_lat", "merch_long")
)
a = (
    np.sin((merchant_lat - lat) / 2) ** 2
    + np.cos(lat) * np.cos(merchant_lat) * np.sin((merchant_lon - lon) / 2) ** 2
)
transactions["distance_km"] = 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

age_bins = [0, 20, 30, 40, 50, 60, 70, 80, np.inf]
age_labels = ["Under 20", "20-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
transactions["age_band"] = pd.cut(
    transactions["age"], age_bins, labels=age_labels, right=False,
)
if transactions["age_band"].isna().any():
    raise ValueError("Every transaction must fit one of the age bands.")

numeric = ["amt", "hour", "distance_km"]
categorical = ["category", "day_of_week"]
features = transactions[numeric + categorical + ["age", "age_band"]]
train, test, y_train, y_test = train_test_split(
    features, transactions["is_fraud"], test_size=0.20, random_state=42,
    stratify=transactions["is_fraud"],
)
print(f"Training transactions: {len(train):,}")
print(f"Held-out transactions: {len(test):,}, including {y_test.sum():,} fraud cases")

Training transactions: 1,037,340
Held-out transactions: 259,335, including 1,501 fraud cases


## Matched alert volume

Each model uses the settings from the final XGBoost notebook and trains on the same 80% split. The simulated review budget is 0.6% of held-out transactions. Each model alerts on its highest scores up to that budget. The comparison represents batch ranking under a fixed review capacity. Rows with tied scores are ordered by their fixed position in the held-out split.

In [2]:
variants = {
    "Exact age": (["amt", "hour", "age", "distance_km"], categorical),
    "Age band": (numeric, categorical + ["age_band"]),
    "No age": (numeric, categorical),
}
alert_fraction = 0.006
alert_count = round(alert_fraction * len(test))
actual_fraud = y_test.to_numpy() == 1
predictions = {}
overall_rows = []

for name, (numeric_columns, categorical_columns) in variants.items():
    columns = numeric_columns + categorical_columns
    preprocessor = ColumnTransformer([
        ("numeric", StandardScaler(), numeric_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ])
    model = XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        random_state=42, n_jobs=4,
    )
    scorer = make_pipeline(preprocessor, model)
    scorer.fit(train[columns], y_train)
    scores = scorer.predict_proba(test[columns])[:, 1]

    if name == "Age band":
        # Exercise the scoring path with a supplied band and no birth date or exact age.
        example = test[columns].iloc[[0]].copy()
        example["age_band"] = example["age_band"].astype(str)
        np.testing.assert_allclose(scorer.predict_proba(example)[0, 1], scores[0])
        age_band_scorer = scorer
        age_band_inputs = columns

    order = np.argsort(-scores, kind="stable")
    alerted = np.zeros(len(test), dtype=bool)
    alerted[order[:alert_count]] = True
    assert alerted.sum() == alert_count
    predictions[name] = alerted

    caught = np.count_nonzero(alerted & actual_fraud)
    overall_rows.append({
        "Model": name,
        "Alerts": alert_count,
        "Frauds caught": caught,
        "False alerts": alert_count - caught,
        "Precision (%)": 100 * caught / alert_count,
        "Recall (%)": 100 * caught / actual_fraud.sum(),
        "PR-AUC (average precision)": average_precision_score(actual_fraud, scores),
    })
    print(f"Evaluated {name}", flush=True)

overall = pd.DataFrame(overall_rows).set_index("Model")
display(overall.round({"Precision (%)": 2, "Recall (%)": 2, "PR-AUC (average precision)": 4}))

Evaluated Exact age
Evaluated Age band
Evaluated No age


,Alerts,Frauds caught,False alerts,Precision (%),Recall (%),PR-AUC (average precision)
Model,,,,,,
Exact age,1556,1280,276,82.26,85.28,0.9161
Age band,1556,1245,311,80.01,82.94,0.8948
No age,1556,1169,387,75.13,77.88,0.8402


## Age-group outcomes

Use the four broader audit groups from the age-removal notebook for every model. Exact age is used here to define common audit groups, outside the scoring path. False-positive rate is the share of legitimate transactions flagged. Recall is the share of fraud transactions flagged. The gap table measures the difference between the highest and lowest group rates in percentage points. Inspect group rates and denominators alongside the gaps, since a gap can shrink when outcomes worsen for a previously better-performing group.

In [3]:
audit_bins = [0, 25, 40, 60, np.inf]
audit_labels = ["Under 25", "25-39", "40-59", "60+"]
audit_groups = pd.cut(
    test["age"], audit_bins, labels=audit_labels, right=False,
).to_numpy()
group_rows = []

for group in audit_labels:
    legitimate = (audit_groups == group) & ~actual_fraud
    fraudulent = (audit_groups == group) & actual_fraud
    assert legitimate.any() and fraudulent.any()
    row = {
        "Age group": group,
        "Legitimate": legitimate.sum(),
        "Fraud": fraudulent.sum(),
    }
    for name, alerted in predictions.items():
        row[f"{name} FPR (%)"] = 100 * alerted[legitimate].mean()
        row[f"{name} recall (%)"] = 100 * alerted[fraudulent].mean()
    group_rows.append(row)

groups = pd.DataFrame(group_rows).set_index("Age group")
assert groups["Fraud"].sum() == actual_fraud.sum()
assert groups["Legitimate"].sum() == (~actual_fraud).sum()
display(groups.round(3))

gaps = pd.DataFrame([
    {
        "Model": name,
        "FPR gap (pp)": groups[f"{name} FPR (%)"].max() - groups[f"{name} FPR (%)"].min(),
        "Recall gap (pp)": groups[f"{name} recall (%)"].max() - groups[f"{name} recall (%)"].min(),
    }
    for name in variants
]).set_index("Model")
display(gaps.round(3))

,Legitimate,Fraud,Exact age FPR (%),Exact age recall (%),Age band FPR (%),Age band recall (%),No age FPR (%),No age recall (%)
Age group,,,,,,,,
Under 25,24272,144,0.128,83.333,0.128,81.944,0.190,81.250
25-39,82062,405,0.152,77.037,0.166,72.099,0.172,69.383
40-59,97669,564,0.116,86.348,0.121,85.106,0.140,79.433
60+,53831,388,0.013,93.041,0.048,91.495,0.117,83.247


,FPR gap (pp),Recall gap (pp)
Model,,
Exact age,0.139,16.004
Age band,0.117,19.396
No age,0.072,13.865


## What changed

At 1,556 alerts, age banding caught 1,245 frauds, 35 fewer than exact age and 76 more than no age. The false-positive gap narrowed from 0.139 to 0.117 percentage points. All four groups had equal or higher false-positive rates with age bands, with the largest increase in the 60+ group. Recall fell in every group, with the largest drop among ages 25-39. The recall gap widened from 16.004 to 19.396 percentage points.

The experiment supports accepting age bands when reducing the precision of scoring inputs is a requirement. The observed group outcomes give mixed evidence about disparities. To try a new input after training, pass a DataFrame with the columns in `age_band_inputs` to `age_band_scorer.predict_proba`. Supply `age_band` as one of the labels listed above.

The split is stratified by transaction and can contain transactions from the same card in training and test. The point estimates describe this synthetic holdout and review budget. A future-period or card-disjoint evaluation would show whether the observed differences persist under other test conditions.